# **Understanding the Problem Statement**

## 1. Problem Definition Document

### 1.1 Background & Motivation

Stroke is a low-prevalence but high-impact medical event. Delayed identification significantly increases mortality and long-term disability. In clinical practice, expert attention is a scarce resource, and reviewing every patient with equal priority is operationally infeasible.

This project aims to **augment clinical triage** by identifying patients at elevated risk of stroke using routinely collected demographic and health indicators.

The system is designed as a decision support tool, not an autonomous diagnostic system.


### 1.2 Decision to Be Supported (Critical)

> **Which patients should be prioritized for expert stroke evaluation?**

The output of the system influences:

- clinical review priority
- follow-up testing
- escalation to specialists

### 1.3 Why This Is Not a “Prediction” Problem

- Stroke prevalence in the dataset is ~5%

- The data is observational and static

- No imaging, lab time-series, or neurological exams

Therefore:

- Perfect classification is **not achievable**

- The goal is **risk stratification**, not diagnosis

## 2. Data Reality & Signal Constraints

### 2.1 Dataset Characteristics

- Total samples: ~5,000

- Positive stroke cases: ~250 (≈5%)

- Severe class imbalance

- Tabular, non-temporal features

**Implication**

- This dataset contains **weak but actionable signal**.

- The signal-to-noise ratio makes **accuracy an invalid metric**.

### 2.2 Modeling Implications

- Accuracy >95% can be achieved by predicting “no stroke” always → useless

- Precision-Recall trade-off dominates model design

- Recall on stroke class is the primary metric

## 3. Objective Function & Success Criteria

### 3.1 Primary Objective

> **Maximize recall for stroke cases while reducing total patients requiring expert review.**

### 3.2 Quantitative Success Criteria

| Metric           | Target             | Rationale                         |
| ---------------- | ------------------ | --------------------------------- |
| Stroke Recall    | ≥ 0.95             | Missing stroke cases is high-risk |
| Review Reduction | ≥ 70%              | Operational efficiency            |
| Precision        | Optimized post-hoc | Controlled via threshold          |
| Calibration      | Stable             | Risk scores must be interpretable |


### 3.3 Interpreting Metrics in Real Terms

Given:

- 5,000 patients

- 250 stroke cases

Target operating point:

- Recall = 0.95 → ~238 strokes detected

- Predicted positives ≈ 350–450 patients

**Impact:**

- Experts review ~8× fewer patients

- Majority of stroke cases prioritized early

This represents a **clinically meaningful improvement**, even with false positives.

## 4. Cost-Sensitive Framing (Very Important)

### 4.1 Error Asymmetry

| Error Type     | Cost                      |
| -------------- | ------------------------- |
| False Negative | Very High (missed stroke) |
| False Positive | Moderate (extra review)   |

Therefore:

- Loss functions are recall-weighted

- Threshold selection prioritizes sensitivity

### 4.2 Why Precision ≠ 0.99

At 5% prevalence:

- Precision >0.9 implies extreme conservatism

- This would **miss many true strokes**

- Clinically unacceptable

High recall + controlled false positives is **the correct trade-off**.

## 5. Human-in-the-Loop Design

### 5.1 Operational Flow

1. Model generates risk scores

2. Threshold selects high-risk cohort

3. Experts review:

    - Borderline predictions

    - Disagreement cases

4. Expert decisions are logged

5. Feedback used for:

    - Threshold recalibration

    - Model retraining

### 5.2 Ethical & Clinical Safeguards

- Model does not override clinicians

- All predictions are explainable

- Human decision always final

> **The system augments clinical judgment rather than replacing it.**

## 6. Modeling Strategy (High-Level)

### 6.1 Pipeline Design

1. Data validation & correction

2. Missing value handling (custom imputers)

3. Feature engineering

4. Class imbalance handling (SMOTE / class-weights)

5. Model training

6. Threshold optimization

7. Explainability layer

### 6.2 Model Selection Philosophy

- Start simple (Logistic / Tree-based)

- Prefer interpretability

- Non-linear models only if recall gains are significant

## 7. MLflow Experiment Strategy

### 7.1 What Is Logged

**For every run:**

- Model version

- Preprocessing pipeline

- Threshold

- Recall / Precision / PR-AUC

- Confusion matrix

- Calibration curves

### 7.2 Experiment Taxonomy

| Experiment Type     | Purpose               |
| ------------------- | --------------------- |
| Baseline            | Understand raw signal |
| Feature Engineering | Improve separability  |
| Threshold Sweeps    | Cost optimization     |
| Bias Analysis       | Fairness validation   |
| Final Model         | Deployment candidate  |


### 7.3 Why Pipelines Are Logged

- Reproducibility

- Safe retraining

- Auditable predictions

- Model governance readiness

## 8. Bias & Robustness Checks

### 8.1 Subgroup Analysis

Evaluate recall across:

- Age groups

- Gender

- Hypertension status

- Smoking history

Goal:

> **No subgroup recall degradation > 5%**

### 8.2 Monitoring in Production

- Stroke recall drift

- Population shift

- Feature distribution change

- Threshold effectiveness


## 9. Deployment Framing (Conceptual)

### 9.1 Model Output

- Probability score (not binary)

- Confidence band

- Top contributing features

### 9.2 Integration Point

- Pre-screening layer

- EMR / reporting dashboard

- Batch inference (daily)

## 10. Explicit Non-Goals (This Is Critical)

- Not a diagnostic device

- Not replacing imaging or neurologists

- Not a medical recommendation engine